# Feature Extraction 08: Sentence-Embedding Text Representation

Encodes the **text** of every CRSP-matched StockTwits message with a pretrained
sentence-transformer and mean-pools the message vectors to the stock-day level. The output
is a compact table of one 384-dimensional vector per (`symbol`, `date`) plus the number of
messages behind it.

**This notebook produces raw material, not model inputs.** Nothing here is merged into
`features_master.pkl` / `merged_master.pkl`, and no return label is used. How the 384
dimensions reach the predictive models -- unchanged, compressed to a few principal
components, or turned into a walk-forward supervised text score -- is decided downstream in
`02 - prepare training dataset/add_text_features.ipynb`, which reads this notebook's output.

**Method**
1. Build the message universe from `merged_with_crsp_mlcrowd/` (message_id, trading `date`,
   and every `symbol` the message is matched to). A message that mentions several tickers
   is kept for *each* of them, exactly as `features_01`/`02`/`04` count it.
2. Stream the raw `messages/` text files once and join each message body into its year
   bucket (checkpointed per source file, Section 3).
3. Encode each message **once** with `all-MiniLM-L6-v2` (384-dim, L2-normalised), then
   fan the vector out to every symbol the message mentions and sum per (`symbol`, `date`).
4. Divide by the message count to get the mean-pooled stock-day vector.

**Scale warning -- read Section 7 before running Section 8.** Encoding is CPU-bound unless a
CUDA GPU is available. Section 7 measures the real throughput on the smallest year and
extrapolates. Measured on this machine (CPU, 24 threads, September 2026): 530-660
messages/sec, i.e. roughly 1.5-2 days for the ~75M-message corpus; the join pass over the
52 GB of raw text takes ~11 minutes. Sections 3 and 8 are both checkpointed so the notebook
can be stopped and resumed across sessions.

## 1. Setup and Configuration

In [ ]:
import html
import json
import math
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
CODE_DIR     = Path(r"C:\Users\skazempour\Documents\StockTwits\Code")
DATA_DIR     = Path(r"C:\Users\skazempour\Documents\StockTwits\Data\v1\data\csv")
MESSAGES_DIR = DATA_DIR / "messages"                  # message_id, message_body (205 files, NOT year-chunked)
RETURNS_DIR  = DATA_DIR / "merged_with_crsp_mlcrowd"  # message-level, per year: message_id, symbol, date

# All output lives OUTSIDE features_mlcrowd/ on purpose: merge_all_feature_files.ipynb globs
# every *.pkl in that folder, and a 384-column table must not be outer-merged into
# features_master.pkl. The downstream builder (add_text_features.ipynb) reads from here.
EMBED_DIR      = DATA_DIR / "text_embeddings_mlcrowd"
JOINED_DIR     = EMBED_DIR / "joined_by_year"      # per-year csv: message_id, date, message_body (ONE row per message)
SYMBOLS_DIR    = EMBED_DIR / "symbols_by_year"     # per-year pkl: message_id, symbol (one row per message-symbol pair)
YEAR_EMBED_DIR = EMBED_DIR / "stock_day_by_year"   # per-year pkl: aggregated stock-day vectors (Section 8 checkpoints)
MSG_EMBED_DIR  = EMBED_DIR / "message_level"       # optional: per-message float16 vectors (SAVE_MESSAGE_EMBEDDINGS)
for d in (JOINED_DIR, SYMBOLS_DIR, YEAR_EMBED_DIR, MSG_EMBED_DIR):
    d.mkdir(parents=True, exist_ok=True)

UNIVERSE_FILE = EMBED_DIR / "message_universe.pkl"            # Section 2 cache
JOIN_PASS_LOG = EMBED_DIR / "join_pass_processed_files.txt"   # Section 3 checkpoint log
OUTPUT_FILE   = EMBED_DIR / "text_embeddings_stock_day.pkl"   # Section 10 final table
META_FILE     = EMBED_DIR / "text_embeddings_meta.json"

# =============================================================================
# PARAMETERS
# =============================================================================
EMBED_MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM         = 384
EMBED_COLS        = [f"embed_{i:03d}" for i in range(EMBED_DIM)]
KEY_COLS          = ["symbol", "date"]
ENCODE_BATCH_SIZE = 256        # forward-pass batch inside model.encode()
ENCODE_CHUNK_ROWS = 200_000    # messages encoded into memory at once (caps peak RAM per year)
REDUCE_ROWS       = 1_000_000  # fold partial per-chunk sums together once they exceed this many rows
CSV_CHUNK_SIZE    = 200_000    # records per chunk when streaming the raw messages/ files
MIN_ID_DIGITS     = 5          # a line starting with fewer digits + comma is body text, not a record
MAX_ID_DIGITS     = 12         # ... and so is one starting with more (every universe message_id has 7-9 digits)

# Optional: also keep every message's own vector (float16, ~0.8 KB/message, ~45 GB for the
# full corpus). Lets a future aggregation (e.g. bullish-only messages, medians, message-level
# models) reuse the encoding instead of repeating days of compute. Off by default for disk.
SAVE_MESSAGE_EMBEDDINGS = False

# Set to a list of years (e.g. [2010]) to smoke-test Section 8 on those years only.
# None processes every year. Section 10 refuses to save while this is set.
TEST_YEARS_ONLY = None

print(f"Messages dir : {MESSAGES_DIR}")
print(f"Returns dir  : {RETURNS_DIR}")
print(f"Embed dir    : {EMBED_DIR}")
print(f"Output file  : {OUTPUT_FILE}")
print(f"Embed model  : {EMBED_MODEL_NAME} ({EMBED_DIM}-dim)")
if TEST_YEARS_ONLY is not None:
    print(f"TEST_YEARS_ONLY = {TEST_YEARS_ONLY}: Section 8 will only process these years")

## 2. Build the Message Universe (cached)

`merged_with_crsp_mlcrowd/` already restricts to Bullish/Bearish-labelled, CRSP-matched
messages and is chunked by year, but has no text. One pass over its (`message_id`, `date`,
`symbol`) columns yields two things:

- **`symbols_{year}.pkl`** -- every (message, symbol) pair. A message mentioning three
  tickers has three rows here and must contribute its vector to all three stock-days.
- **`universe`** -- one row per message (`message_id` -> `date`, `year`), used to route
  message bodies to year buckets in Section 3 and to decide what is worth encoding.

Each message has exactly one trading date (asserted, not assumed), so the message is a
clean unit for encoding and the pair table handles the fan-out at aggregation time.

In [ ]:
RETURN_FILES = sorted(RETURNS_DIR.glob("stocktwits_crsp_*.csv"))
print(f"Found {len(RETURN_FILES)} year files in {RETURNS_DIR.name}")

if UNIVERSE_FILE.exists() and len(list(SYMBOLS_DIR.glob("symbols_*.pkl"))) == len(RETURN_FILES):
    universe = pd.read_pickle(UNIVERSE_FILE)
    print(f"Loaded cached universe: {len(universe):,} messages "
          f"({universe['year'].min()}-{universe['year'].max()})")
else:
    universe_frames = []
    for f in tqdm(RETURN_FILES, desc="Building message universe"):
        year = int(f.stem.split("_")[-1])
        df_y = pd.read_csv(f, usecols=["message_id", "date", "symbol"],
                           dtype={"message_id": "int64", "symbol": "string"})
        df_y["date"] = pd.to_datetime(df_y["date"])

        # One trading date per message -- the whole design relies on it
        n_dates = df_y.groupby("message_id")["date"].nunique()
        assert n_dates.max() == 1, f"{year}: {int((n_dates > 1).sum())} message_ids map to >1 date"

        # (message, symbol) pairs: the fan-out table used in Section 5
        pairs = df_y[["message_id", "symbol"]].drop_duplicates().reset_index(drop=True)
        pairs.to_pickle(SYMBOLS_DIR / f"symbols_{year}.pkl")

        msgs = df_y[["message_id", "date"]].drop_duplicates("message_id")
        msgs["year"] = np.int16(year)
        universe_frames.append(msgs)
        print(f"  {year}: {len(df_y):,} message-symbol rows -> {len(msgs):,} messages, "
              f"{len(pairs):,} pairs ({(len(pairs) / len(msgs) - 1):.1%} extra from multi-symbol messages)")
        del df_y, pairs, msgs

    universe = pd.concat(universe_frames, ignore_index=True)
    del universe_frames
    n_dup = int(universe["message_id"].duplicated().sum())
    assert n_dup == 0, f"{n_dup} message_ids appear in more than one year file"
    universe = universe.set_index("message_id").sort_index()
    universe.to_pickle(UNIVERSE_FILE)

print(f"\nUniverse: {len(universe):,} messages across {universe['year'].nunique()} years")
print(universe.groupby("year").size().to_string())

## 3. Join Message Text into Year Buckets (checkpointed)

`messages/` is **not** year-chunked: its 205 files carry `message_id` ranges that each span
many calendar years. This cell streams every `msg_*.csv` once, inner-joins each chunk's
`message_id` against the universe, and appends the matches to `joined_{year}.csv` --
one row per message (the symbol fan-out is applied later from `symbols_{year}.pkl`).

Finished source files are recorded in `JOIN_PASS_LOG`, so re-running after completion is a
no-op and an interruption costs at most one file. A file's matches are only written once the
whole file has parsed, so a parser fallback (below) can never produce duplicate rows.

**Reader note.** The files are two-column CSVs (`message_id`, `message_body`) whose bodies
contain newlines, commas and stray or unterminated quote characters. A CSV tokenizer is the
wrong tool: pandas' C engine aborts on `msg_000.csv` with "Buffer overflow caught" after
600k rows, and an unterminated quote can make either engine silently swallow every record
that follows it. `iter_message_records()` instead reads physical lines and treats every line
that begins with an id of `MIN_ID_DIGITS`..`MAX_ID_DIGITS` digits followed by a comma as the
start of a record; any other line (including body lines that start with a short or absurdly
long number) continues the previous body. Nothing is interpreted inside a body,
so a stray quote cannot derail the stream. (Record counts per file are reported so the
reader can be checked against `grep -c '^[0-9]*,'`.)

The files are **not** ordered by id either: `msg_000.csv` runs from id 4 to id ~103M in its
first 640k rows, so a year's messages are spread across all 205 files and the whole set must
be scanned once.

In [ ]:
def _load_processed_files():
    return set(JOIN_PASS_LOG.read_text().splitlines()) if JOIN_PASS_LOG.exists() else set()

def _mark_processed(fname):
    with open(JOIN_PASS_LOG, "a") as fh:
        fh.write(fname + "\n")

def _unquote(body):
    """Undo CSV quoting of a body captured verbatim from the file."""
    body = body.rstrip("\r\n")
    if len(body) >= 2 and body[0] == '"' and body[-1] == '"':
        body = body[1:-1].replace('""', '"')
    return body

def _records_frame(ids, bodies):
    return pd.DataFrame({"message_id": np.asarray(ids, dtype="int64"),
                         "message_body": [_unquote("".join(b)) for b in bodies]})

def iter_message_records(path, chunk_rows=CSV_CHUNK_SIZE):
    """Stream (message_id, message_body) records from one messages/ file as DataFrames of
    up to chunk_rows rows. A record starts at every physical line that begins with an id of
    MIN_ID_DIGITS..MAX_ID_DIGITS digits followed by a comma; every other line continues the previous body
    (multi-line bodies are CSV-quoted in the files, and the quotes are removed by _unquote)."""
    ids, bodies, cur_id, cur = [], [], None, []
    with open(path, encoding="utf-8", errors="replace", newline="") as fh:
        fh.readline()  # header: message_id,message_body
        for line in fh:
            if line[:1].isdigit():
                i = line.find(",")
                if MIN_ID_DIGITS <= i <= MAX_ID_DIGITS and line[:i].isdigit():
                    if cur_id is not None:
                        ids.append(cur_id)
                        bodies.append(cur)
                        if len(ids) >= chunk_rows:
                            yield _records_frame(ids, bodies)
                            ids, bodies = [], []
                    cur_id, cur = int(line[:i]), [line[i + 1:]]
                    continue
            if cur_id is not None:
                cur.append(line)
        if cur_id is not None:
            ids.append(cur_id)
            bodies.append(cur)
        if ids:
            yield _records_frame(ids, bodies)

def _join_one_file(path, universe):
    """All universe matches in one messages/ file: DataFrame(message_id, date, year, message_body),
    plus the number of records read from the file."""
    matches, n_records = [], 0
    for chunk in iter_message_records(path):
        n_records += len(chunk)
        joined = chunk.join(universe, on="message_id", how="inner")
        if not joined.empty:
            matches.append(joined)
    if not matches:
        return pd.DataFrame(columns=["message_id", "date", "year", "message_body"]), n_records
    return pd.concat(matches, ignore_index=True), n_records

message_files = sorted(MESSAGES_DIR.glob("msg_*.csv"))
processed = _load_processed_files()
todo_files = [f for f in message_files if f.name not in processed]
print(f"messages/ files: {len(message_files)} total, {len(processed)} already joined, {len(todo_files)} remaining")

join_stats = []
for f in tqdm(todo_files, desc="Joining messages/ files"):
    t0 = time.time()
    matched, n_records = _join_one_file(f, universe)
    for year, grp in matched.groupby("year"):
        out_path = JOINED_DIR / f"joined_{int(year)}.csv"
        grp[["message_id", "date", "message_body"]].to_csv(
            out_path, mode="a", header=not out_path.exists(), index=False, lineterminator="\n")
    _mark_processed(f.name)
    join_stats.append({"file": f.name, "records": n_records, "matched": len(matched),
                       "secs": round(time.time() - t0, 1)})

if join_stats:
    js = pd.DataFrame(join_stats)
    print(f"Files read: {len(js)}; records: {js['records'].sum():,}; matched: {js['matched'].sum():,}; "
          f"time: {js['secs'].sum() / 60:.1f} min")
    display(js.describe().round(1))
print("Join pass complete (or already complete from a prior run).")
print("Joined year files:", sorted(p.name for p in JOINED_DIR.glob("joined_*.csv")))

## 4. Core Functions: Text Cleaning, Encoding, Fan-out Aggregation

`aggregate_chunk()` is the one place the multi-symbol logic lives: each message is encoded
once, then its vector is repeated for every symbol in the year's pair table before the
per-(`symbol`, `date`) **sum** is taken. Sums (not per-chunk means) are accumulated across
chunks and divided by the count only at the end, so the result is exact regardless of how a
year is split into chunks. Sums are kept in float64 and cast to float32 on output.

In [ ]:
WHITESPACE_RE = re.compile(r"\s+")

def clean_text(body):
    """Unescape HTML entities and collapse whitespace. Cashtags/mentions are kept as
    context for the embedding model rather than stripped."""
    return WHITESPACE_RE.sub(" ", html.unescape(str(body))).strip()

def load_year_messages(year):
    df = pd.read_csv(JOINED_DIR / f"joined_{year}.csv",
                     dtype={"message_id": "int64", "message_body": "string"})
    n_dup = int(df["message_id"].duplicated().sum())
    assert n_dup == 0, f"{year}: {n_dup} duplicate message_ids in joined file (re-run Section 3 from a clean JOINED_DIR)"
    df["date"] = pd.to_datetime(df["date"])
    df["message_body"] = df["message_body"].fillna("").map(clean_text)
    # Sorting by date keeps every encode chunk's partial sums small (few distinct stock-days per chunk)
    return df.sort_values(["date", "message_id"]).reset_index(drop=True)

def load_year_symbols(year):
    return pd.read_pickle(SYMBOLS_DIR / f"symbols_{year}.pkl")

def embed_texts(model, texts, batch_size=ENCODE_BATCH_SIZE):
    return model.encode(list(texts), batch_size=batch_size, show_progress_bar=False,
                        normalize_embeddings=True, convert_to_numpy=True).astype("float32")

def aggregate_chunk(sub, embeddings, symbols):
    """Partial per-(symbol, date) SUMS for one chunk of messages.
    sub        : DataFrame slice with message_id, date (rows align with `embeddings`)
    embeddings : (len(sub), EMBED_DIM) array
    symbols    : the year's (message_id, symbol) pair table
    A message's vector is fanned out to every symbol it is paired with."""
    fan = pd.DataFrame({"pos": np.arange(len(sub)),
                        "message_id": sub["message_id"].to_numpy(),
                        "date": sub["date"].to_numpy()})
    fan = fan.merge(symbols, on="message_id", how="inner")
    out = pd.DataFrame(embeddings[fan["pos"].to_numpy()].astype(np.float64), columns=EMBED_COLS)
    out["symbol"] = fan["symbol"].to_numpy()
    out["date"] = fan["date"].to_numpy()
    out["embed_n"] = 1
    return out.groupby(KEY_COLS, as_index=False, sort=False).sum()

def reduce_partials(partials):
    """Fold a list of partial-sum tables into one (same schema, one row per stock-day)."""
    if len(partials) == 1:
        return partials[0]
    return pd.concat(partials, ignore_index=True).groupby(KEY_COLS, as_index=False, sort=False).sum()

def finalize_aggregation(partials):
    """Sums -> means. Output: symbol, date, embed_n, embed_000..embed_383 (float32)."""
    totals = reduce_partials(partials)
    n = totals["embed_n"].to_numpy()[:, None]
    totals[EMBED_COLS] = (totals[EMBED_COLS].to_numpy() / n).astype(np.float32)
    totals["embed_n"] = totals["embed_n"].astype("int32")
    return totals[KEY_COLS + ["embed_n"] + EMBED_COLS].sort_values(KEY_COLS).reset_index(drop=True)

## 5. Per-Year Driver

For year `Y`: load the joined text, encode in row-chunks of `ENCODE_CHUNK_ROWS`, fan out and
sum each chunk immediately, discard the embeddings, and fold partial sums together whenever
they exceed `REDUCE_ROWS` rows so peak memory is bounded by the number of stock-days in the
year, not by the number of messages.

`SentenceTransformer.encode()` accumulates every embedding it produces into one array before
returning; for a 25M-message year that is a 36 GiB float32 array, which is what crashed the
first attempt at 2021. Chunking at this outer level caps peak memory.

Aggregation has no cross-year state and no target label, so years can be processed in any
order; processing them in order is just a convenient way to checkpoint (Section 8).

In [ ]:
def process_year(year, model, chunk_rows=ENCODE_CHUNK_ROWS, save_message_level=SAVE_MESSAGE_EMBEDDINGS):
    df = load_year_messages(year)
    symbols = load_year_symbols(year)
    n = len(df)
    n_chunks = math.ceil(n / chunk_rows)

    partials, partial_rows = [], 0
    encode_secs = 0.0
    n_pairs = 0

    for chunk_i, start in enumerate(range(0, n, chunk_rows)):
        end = min(start + chunk_rows, n)
        sub = df.iloc[start:end]

        t0 = time.time()
        embeddings = embed_texts(model, sub["message_body"].tolist())
        encode_secs += time.time() - t0

        if save_message_level:
            np.save(MSG_EMBED_DIR / f"msg_{year}_{chunk_i:04d}_ids.npy", sub["message_id"].to_numpy())
            np.save(MSG_EMBED_DIR / f"msg_{year}_{chunk_i:04d}_vec.npy", embeddings.astype(np.float16))

        part = aggregate_chunk(sub, embeddings, symbols)
        n_pairs += int(part["embed_n"].sum())
        partials.append(part)
        partial_rows += len(part)
        del embeddings

        if partial_rows > REDUCE_ROWS:
            partials = [reduce_partials(partials)]
            partial_rows = len(partials[0])

        if n_chunks > 1:
            print(f"    {year}: chunk {chunk_i + 1}/{n_chunks} ({end:,}/{n:,} messages) "
                  f"cumulative encode time {encode_secs:,.0f}s")

    agg = finalize_aggregation(partials)
    stats = {"year": year, "n_messages": n, "n_pairs": n_pairs, "n_stock_days": len(agg),
             "n_chunks": n_chunks, "encode_secs": round(encode_secs, 1),
             "msgs_per_sec": round(n / max(encode_secs, 1e-9), 1)}
    return agg, stats

## 6. Validate on the Smallest Year

Runs the real pipeline end-to-end (model load, chunked encode, fan-out aggregation) on the
smallest joined year and checks the output against independent computations:

- schema and dtypes;
- `embed_n` sums to the number of (message, symbol) pairs that have text -- i.e. the
  fan-out is neither dropping multi-symbol messages nor double-counting;
- every mean vector has norm in (0, 1] (means of unit vectors), and single-message
  stock-days have norm 1;
- a brute-force recomputation of a few stock-days (re-encode their messages, average by hand)
  matches the pipeline output.

In [ ]:
import os
# The py313 conda env ships numpy/scikit-learn with Intel's OpenMP runtime and pip's torch
# wheel bundles its own copy; importing both in one process aborts with "OMP: Error #15"
# unless the duplicate is allowed. Must be set before torch is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}, device = {DEVICE}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

years_available = sorted(int(p.stem.split("_")[-1]) for p in JOINED_DIR.glob("joined_*.csv"))
print(f"Years available after the join pass: {years_available}")
first_year = years_available[0]

agg_first, stats_first = process_year(first_year, embed_model, save_message_level=False)
print(f"\nYear {first_year}: {stats_first}")
display(agg_first.head())

# --- schema ---
assert list(agg_first.columns) == KEY_COLS + ["embed_n"] + EMBED_COLS
assert agg_first[EMBED_COLS].dtypes.eq("float32").all()
assert not agg_first.duplicated(KEY_COLS).any()

# --- fan-out accounting: embed_n must equal the number of (message, symbol) pairs with text ---
msgs_first = load_year_messages(first_year)
pairs_with_text = load_year_symbols(first_year).merge(msgs_first[["message_id", "date"]], on="message_id")
assert int(agg_first["embed_n"].sum()) == len(pairs_with_text), \
    f"embed_n total {int(agg_first['embed_n'].sum()):,} != pairs with text {len(pairs_with_text):,}"
multi = pairs_with_text.groupby("message_id").size()
print(f"Messages with text: {len(msgs_first):,}; pairs: {len(pairs_with_text):,}; "
      f"multi-symbol messages: {int((multi > 1).sum()):,}")

# --- geometry: means of unit vectors ---
norms = np.linalg.norm(agg_first[EMBED_COLS].to_numpy(), axis=1)
assert norms.max() <= 1 + 1e-4 and norms.min() > 0
single = agg_first["embed_n"] == 1
assert np.allclose(norms[single], 1.0, atol=1e-4), "single-message stock-days should be unit vectors"
print(f"Mean-vector norm: min {norms.min():.3f}, median {np.median(norms):.3f}, max {norms.max():.3f}")

# --- brute-force recomputation of a few stock-days, always including one multi-symbol message ---
check_keys = agg_first.loc[(agg_first["embed_n"] > 1) & (agg_first["embed_n"] <= 25), KEY_COLS]
check_keys = check_keys.sample(min(5, len(check_keys)), random_state=0)
multi_ids = multi[multi > 1].index[:1]
if len(multi_ids):
    check_keys = pd.concat([check_keys, pairs_with_text.loc[pairs_with_text["message_id"].isin(multi_ids), KEY_COLS]])
check_keys = check_keys.drop_duplicates()
for _, (sym, day) in check_keys.iterrows():
    ids = pairs_with_text.loc[(pairs_with_text["symbol"] == sym) & (pairs_with_text["date"] == day), "message_id"]
    texts = msgs_first.loc[msgs_first["message_id"].isin(ids), "message_body"]
    ref = embed_texts(embed_model, texts.tolist()).mean(axis=0)
    got = agg_first.loc[(agg_first["symbol"] == sym) & (agg_first["date"] == day), EMBED_COLS].to_numpy()[0]
    assert np.allclose(ref, got, atol=1e-5), f"mismatch on {sym} {day.date()}"
    print(f"  brute-force check {sym:6s} {day.date()}  n={len(texts):3d}  OK")

print("\nValidation passed: chunked encode + symbol fan-out + mean-pooling are consistent.")

## 7. Runtime Benchmark and Full-Corpus Estimate

**Read this before running Section 8.** The throughput below is the real rate measured in
Section 6 on this machine, extrapolated to the full universe from Section 2. It is a lower
bound: later years have far more messages per file, so cache and memory pressure differ.

In [ ]:
total_messages = len(universe)
observed_rate = stats_first["msgs_per_sec"]
est_hours = total_messages / observed_rate / 3600

print(f"Device: {DEVICE}")
print(f"Observed encoding rate ({first_year}): {observed_rate:,.1f} messages/sec")
print(f"Messages in universe: {total_messages:,}")
print(f"Estimated full-corpus encoding time: {est_hours:,.1f} hours (~{est_hours / 24:,.1f} days)")
print()
stock_days_est = total_messages / max(stats_first["n_messages"], 1) * stats_first["n_stock_days"]
print(f"Stock-day output size (float32): ~{stock_days_est * EMBED_DIM * 4 / 1024**3:,.1f} GB (rough extrapolation)")
if SAVE_MESSAGE_EMBEDDINGS:
    print(f"Message-level embeddings (float16): ~{total_messages * EMBED_DIM * 2 / 1024**3:,.1f} GB under {MSG_EMBED_DIR}")
print()
print("Section 8 is checkpointed per year under YEAR_EMBED_DIR: it is safe to run for a while,")
print("stop, and resume later across multiple sessions.")

## 8. Process All Years (checkpointed)

Resumable at year granularity: any year whose file already exists under `YEAR_EMBED_DIR` is
skipped. There is no cross-year state, so resuming is simply "skip what is on disk".

In [ ]:
def _year_embed_path(year):
    return YEAR_EMBED_DIR / f"stock_day_{year}.pkl"

years_available = sorted(int(p.stem.split("_")[-1]) for p in JOINED_DIR.glob("joined_*.csv"))
if TEST_YEARS_ONLY is not None:
    years_available = [y for y in years_available if y in TEST_YEARS_ONLY]
    print(f"TEST_YEARS_ONLY active: restricting this run to {years_available}")

done_years = sorted(int(p.stem.split("_")[-1]) for p in YEAR_EMBED_DIR.glob("stock_day_*.pkl"))
remaining_years = [y for y in years_available if y not in done_years]
print(f"Years already done: {done_years}")
print(f"Years remaining:    {remaining_years}")

run_stats = []
for year in tqdm(remaining_years, desc="Encoding years"):
    agg, stats = process_year(year, embed_model)
    agg.to_pickle(_year_embed_path(year))
    run_stats.append(stats)
    print(f"  {year}: {stats['n_messages']:,} msgs / {stats['n_pairs']:,} pairs -> "
          f"{stats['n_stock_days']:,} stock-days ({stats['msgs_per_sec']:.1f} msg/s)")

print("\nAll years processed (or resumed to completion).")
if run_stats:
    display(pd.DataFrame(run_stats))

## 9. Combine All Years and Inspect

In [ ]:
year_files = sorted(YEAR_EMBED_DIR.glob("stock_day_*.pkl"))
years_done = [int(p.stem.split("_")[-1]) for p in year_files]
years_expected = sorted(int(p.stem.split("_")[-1]) for p in SYMBOLS_DIR.glob("symbols_*.pkl"))
missing_years = sorted(set(years_expected) - set(years_done))

features_all = pd.concat([pd.read_pickle(f) for f in year_files], ignore_index=True)
features_all = features_all.sort_values(KEY_COLS).reset_index(drop=True)
assert not features_all.duplicated(KEY_COLS).any()

print(f"Shape: {features_all.shape}  (~{features_all.memory_usage(deep=True).sum() / 1024**3:.2f} GB in memory)")
print(f"Years: {years_done}")
print(f"Date range: {features_all['date'].min().date()} to {features_all['date'].max().date()}")
print(f"Unique symbols: {features_all['symbol'].nunique():,}")
print(f"Messages per stock-day: median {features_all['embed_n'].median():.0f}, "
      f"mean {features_all['embed_n'].mean():.1f}, max {features_all['embed_n'].max():,}")
print(f"Nulls: {int(features_all.isnull().sum().sum())}")
if missing_years:
    print(f"\nWARNING: years without output yet: {missing_years} -- Section 10 will refuse to save.")
if TEST_YEARS_ONLY is not None:
    print(f"\nWARNING: TEST_YEARS_ONLY = {TEST_YEARS_ONLY}: partial smoke-test combine, not the full corpus.")

## 10. Save

Refuses to write a partial corpus: every year with a `symbols_{year}.pkl` must have its
stock-day file, and `TEST_YEARS_ONLY` must be `None`.

In [ ]:
assert TEST_YEARS_ONLY is None, f"TEST_YEARS_ONLY = {TEST_YEARS_ONLY}: refusing to save a smoke-test combine over {OUTPUT_FILE.name}"
assert not missing_years, f"years without output: {missing_years}"

print(f"Saving to: {OUTPUT_FILE}")
features_all.to_pickle(OUTPUT_FILE)

meta = {"model": EMBED_MODEL_NAME, "dim": EMBED_DIM, "normalized": True, "pooling": "mean over messages",
        "key": KEY_COLS, "columns": ["embed_n"] + EMBED_COLS, "years": years_done,
        "n_stock_days": int(len(features_all)), "n_message_symbol_pairs": int(features_all["embed_n"].sum()),
        "created": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}
META_FILE.write_text(json.dumps(meta, indent=2))

verify = pd.read_pickle(OUTPUT_FILE)
assert verify.shape == features_all.shape
print(f"Saved and verified. {OUTPUT_FILE.stat().st_size / 1024**2:,.0f} MB; metadata in {META_FILE.name}")

## Summary

**Rationale.** Every other feature in this project treats a message's self-reported
Bullish/Bearish tag as the unit of signal and discards the text. This notebook lets the text
speak through a pretrained sentence embedding, aggregated to the stock-day, as raw material
for testing whether the text carries information beyond the tag.

**Output.** `text_embeddings_mlcrowd/text_embeddings_stock_day.pkl`: one row per
(`symbol`, `date`) with `embed_n` (message-symbol pairs behind the row) and `embed_000` ..
`embed_383` (float32 mean of L2-normalised message vectors). Nothing is fit on returns here.

**Design notes.**
- A message is encoded once and counted toward every symbol it mentions, matching how
  `features_01`/`02`/`04` count messages. Deduplicating on `message_id` (as an earlier draft
  did) silently attributed multi-ticker messages to only their first symbol.
- The output is deliberately kept out of `features_mlcrowd/` so the generic feature merge
  never picks up 384 columns. The downstream builder
  (`02 - prepare training dataset/add_text_features.ipynb`) turns this table into model
  inputs in one of three forms: all 384 dimensions, a few principal components, or a
  walk-forward supervised text score.
- A 0 vector is *not* a neutral embedding, so missing stock-days are not imputed here; the
  builder decides how "no messages" is represented for each form.